<a href="https://colab.research.google.com/github/AdamClarkStandke/LangChainTextInteraction/blob/main/FineTuningLLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets # https://pypi.org/project/datasets/
!pip install evaluate
!pip install sentence-transformers
!pip install setfit

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer
import numpy as np
import datasets
import evaluate
from setfit import sample_dataset, SetFitModel
from setfit import TrainingArguments as SetFitTrainingArguments
from setfit import Trainer as SetFitTrainer

In [ ]:
# Prepare data and splits
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]
# prepare data for few shot learning
sampled_train_data = sample_dataset(tomatoes['train'], num_samples=16)

In [ ]:
# Data for Supervised Fintuning
print(train_data[0])
print(train_data.num_rows)

In [ ]:
# Data for Sentence Transformer FineTuning
print(sampled_train_data[0])
print(sampled_train_data.num_rows)

# Supervised Fine Tuning

In [ ]:
# Load model and tokenizer
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

In [ ]:
def compute_metrics(eval_pred):
   """Calculate F1 score"""
   logits, labels = eval_pred
   predictions = np.argmax(logits, axis=-1)

   load_f1 = evaluate.load("f1")
   f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
   return {"f1": f1}

In [ ]:
# # print layers of pre-trained model!!!!
# for name, param in model.named_parameters():
#   print(f"name={name}, weights={param.shape}")

In [ ]:
# choosing layers to freeze and train!!!!
for name, param in model.named_parameters():
  # trainable portion
  if name.startswith("classifier"):
    param.requires_grad = True
  # non-trainable portion
  else:
    param.requires_grad = False

In [ ]:
# # printing out whether layer is trainable or not
# for name, param in model.named_parameters():
#   print(f"name={name}, trainable={param.requires_grad}")

In [ ]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   processing_class=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
# Load model and tokenizer
model_id = "bert-base-cased"
model_two = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer_two = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# freezing everything before block 11 through indexing!!!
for index, (name, param) in enumerate(model_two.named_parameters()):
    if index < 165:
        param.requires_grad = False
    #print(f"index={index}, name={name}, trainable={param.requires_grad}")


In [ ]:
# Trainer which executes the training process
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_two)
trainer = Trainer(
   model=model_two,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   processing_class=tokenizer_two,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate()

# Few-Shot Classification

In [ ]:
model_three = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

In [ ]:
model_three.model_card_data

In [ ]:
# Define training arguments
args = SetFitTrainingArguments(
    num_epochs=3, # The number of epochs to use for contrastive learning
    num_iterations=20  # The number of text pairs to generate
)
args.eval_strategy = args.evaluation_strategy

# Create trainer
trainer = SetFitTrainer(
    model=model_three,
    args=args,
    train_dataset=sampled_train_data,
    eval_dataset=test_data,
    metric="f1"
)
# Training loop
trainer.train()

In [ ]:
trainer.evaluate()